In [1]:
import os
import numpy as np
from nilearn import image
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import seaborn as sns
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr, zscore, ttest_rel, pearsonr, ttest_ind
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
import seaborn as sns
from scipy.stats import sem
from scipy.signal import detrend
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import pandas as pd
import numpy as np
from bert_score import score
from tqdm import tqdm



#### determine the correlation between BERT score between sentences and difference-scores in terms of moral-nonmoral ratio between sentences


###### per sentence, tunnel: Spearman rho = 0.0025819580364250996 ; p = 0.5633814565050111


In [2]:
foundationScores_dir = "/Users/silvycollin/Documents/GitHub/wholeBrain_narratives_MAC_MFT/text/foundationScores/"

In [3]:
whichVersion = 'persentence' #persentence, 3sec, 6sec

# save
save_dir = os.path.expanduser(
    "~/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/RSA_relToMoralnonmoral/"
)

In [4]:

whichStory = "tunnel"  # tunnel, 21styear

base_dir = os.path.expanduser(
    "~/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/processed_first_level_per_sentence/"
)

if whichVersion =='persentence':
    foundationScores_file = f"{foundationScores_dir}{whichStory}_MFT_MAC.xlsx"
    foundationScores = pd.read_excel(foundationScores_file)
elif whichVersion =='3sec':
    foundationScores_file = f"{foundationScores_dir}{whichStory}_3_seconds_MFT_MAC.xlsx"
    foundationScores = pd.read_excel(foundationScores_file)
elif whichVersion =='6sec':
    foundationScores_file = f"{foundationScores_dir}{whichStory}_6_seconds_MFT_MAC.xlsx"
    foundationScores = pd.read_excel(foundationScores_file)

print(whichStory)


tunnel


In [5]:
# ------------------------
# load foundation scores 
# ------------------------
# start and end column is start and end of the sentences in seconds, TR was 1.5 sec

foundationScoresSelect = foundationScores[["MAC_a_moral_nonmoral","sentence"]]
foundationScoresSelect = foundationScoresSelect.copy()


In [6]:
# select only moral sentences
foundationScoresSelect_nonzero = foundationScoresSelect[foundationScoresSelect["MAC_a_moral_nonmoral"] != 0]


In [7]:
foundationScoresSelect_nonzero

,MAC_a_moral_nonmoral,sentence
0,0.500000,"Tonight, a story by Frederick Pohl. Tunnel Und..."
2,2.000000,It was more real than any dream he had ever ha...
3,0.500000,He could still hear and feel that sharp metal ...
10,1.000000,"Guy, what's wrong?"
15,1.000000,I don't know.
...,...,...
538,1.000000,It came closer.
539,0.500000,The glare was partially blocked.
540,0.200000,"And then, Guy Burkhardt knew that the towering..."
542,1.000000,I took your brain circuits and had them reduced.


In [8]:

# extract the column as a numpy array
vals = foundationScoresSelect_nonzero["MAC_a_moral_nonmoral"].values

# compute all pairwise absolute differences
RDM_moralscores = np.abs(vals[:, None] - vals[None, :])

RDM_moralscores = pd.DataFrame(RDM_moralscores,
                               index=foundationScoresSelect_nonzero.index,
                               columns=foundationScoresSelect_nonzero.index)

In [9]:

# ----------------------------------------------------
# 1. Collect sentences
# ----------------------------------------------------
sentences = foundationScoresSelect_nonzero["sentence"].tolist()
n = len(sentences)

# ----------------------------------------------------
# 2. Load model & tokenizer
# ----------------------------------------------------
model_name = "sentence-transformers/all-mpnet-base-v2" 

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ----------------------------------------------------
# 3. Encode all sentences (mean-pooled embeddings)
# ----------------------------------------------------
def encode(texts, batch_size=32):
    embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding"):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)

        with torch.no_grad():
            output = model(**enc).last_hidden_state

        # Mean pooling
        mask = enc['attention_mask'].unsqueeze(-1)
        pooled = (output * mask).sum(dim=1) / mask.sum(dim=1)
        embeddings.append(pooled.cpu())

    return torch.cat(embeddings, dim=0)

embeddings = encode(sentences)   

# ----------------------------------------------------
# 4. Compute cosine similarity matrix
# ----------------------------------------------------
emb = embeddings / embeddings.norm(dim=1, keepdim=True)  # normalize
similarity_matrix = emb @ emb.T                           # fast 

# ----------------------------------------------------
# 5. Convert similarity → RDM (distance)
# ----------------------------------------------------
RDM = 1 - similarity_matrix.numpy()

# ----------------------------------------------------
# 6. Save as dataframe if desired
# ----------------------------------------------------
RDM_df = pd.DataFrame(
    RDM,
    index=[f"sent_{i}" for i in range(n)],
    columns=[f"sent_{i}" for i in range(n)]
)

RDM_df.to_csv(str(save_dir)+f"{whichVersion}_{whichStory}_RDM_fast_embeddings.csv")

print("RDM shape:", RDM_df.shape)


Encoding: 100%|█████████████████████████████████| 10/10 [00:02<00:00,  3.71it/s]


RDM shape: (317, 317)


In [10]:


mask = np.triu(np.ones(RDM_df.shape), k=1).astype(bool)
RDM_BERT_vec = RDM_df.to_numpy()[mask]

mask = np.triu(np.ones(RDM_moralscores.shape), k=1).astype(bool)
moral_vec = RDM_moralscores.values[mask]

In [11]:

rho, p = spearmanr(RDM_BERT_vec, moral_vec)

print(f"Spearman rho = {rho:.6f} ; p = {p:.10f}")

Spearman rho = 0.067359 ; p = 0.0000000000
